# oCen stream: GPU job runner

One notebook for every GPU job. Edit `JOB` below, then **Runtime → Run all** (A100). The code is installed fresh from `ocen_stream@main`; inputs come from the GitHub release `INPUTS`; logs and checkpoints go to the private repository `ocen_stream_runs` (pushed every 5 minutes and at the end), so a job resumes after a disconnect by running the notebook again.

Needs the Colab secret `GH_RUNS_TOKEN` (fine-grained GitHub token, Contents read/write on `ocen_stream_runs` only).

In [ ]:
JOB = "ocen_stream.smc --inputs /content/runs/inputs/mock_scanbest_20261027 --out /content/runs/smc_mock_scanbest_n300_20261027 --n 300"   # python -m <JOB>; --out under /content/runs
INPUTS = "gpu-inputs-20261027"   # release downloaded to /content/inp (mock inputs live in /content/runs/inputs/...)

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!nproc

In [ ]:
import os, subprocess
from google.colab import userdata
os.environ['GH_RUNS_TOKEN'] = userdata.get('GH_RUNS_TOKEN')                     # never printed
!pip install -q --no-deps --force-reinstall "git+https://github.com/vasilybelokurov/ocen_stream@main"
!pip install -q diffrax
!git ls-remote https://github.com/vasilybelokurov/ocen_stream refs/heads/main
!python -m ocen_stream.runsync clone /content/runs
import jax, cupy
print('jax', jax.__version__, 'cupy', cupy.__version__, jax.devices())
assert jax.devices()[0].platform == 'gpu', 'no GPU: change the runtime type'

In [ ]:
%%bash -s "$INPUTS"
set -e
mkdir -p /content/inp && cd /content/inp
R=https://github.com/vasilybelokurov/ocen_stream/releases/download/$1
for f in host_tables.npz spray_inputs.npz score_inputs.npz points.json SHA256SUMS; do wget -q -N $R/$f; done
sha256sum -c SHA256SUMS

In [ ]:
sync = subprocess.Popen(['python', '-m', 'ocen_stream.runsync', 'loop', '/content/runs', '300'], stdout=open('/content/sync.log', 'a'), stderr=subprocess.STDOUT)
!python -m $JOB
sync.terminate()
!python -m ocen_stream.runsync push /content/runs; tail -3 /content/sync.log